In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

# Intro to tensors

In [2]:
# Working with tensors
a = torch.rand(10, 5)
b = torch.rand(5, 17)
mult = torch.matmul(a, b)
print(mult.shape)

torch.Size([10, 17])


In [3]:
# Working with tensors (batch multiplication)
a = torch.rand(16, 10, 5)
b = torch.rand(16, 5, 17)
mult = torch.matmul(a, b)
print(mult.shape)

torch.Size([16, 10, 17])


In [4]:
# Gradient example
a = torch.Tensor([1, 2, 3])
b = torch.Tensor([4, 5, 6])
c = torch.Tensor([7, 8, 9])

a.requires_grad = True
b.requires_grad = True
c.requires_grad = True

torch.sum((a * b) + c).backward()
print(a.grad), print(b.grad), print(c.grad)

tensor([4., 5., 6.])
tensor([1., 2., 3.])
tensor([1., 1., 1.])


(None, None, None)

In [18]:
a = torch.Tensor(torch.rand(1, 4))
a.requires_grad = True
b = a**2
c = b*2
d = c.mean()
e = c.sum()

In [7]:
print(a), print(b), print(c), print(d), print(e)

tensor([[0.7954, 0.5461, 0.6934, 0.8684]], requires_grad=True)
tensor([[0.6327, 0.2982, 0.4808, 0.7541]], grad_fn=<PowBackward0>)
tensor([[1.2653, 0.5965, 0.9616, 1.5083]], grad_fn=<MulBackward0>)
tensor(1.0829, grad_fn=<MeanBackward0>)
tensor(4.3316, grad_fn=<SumBackward0>)


(None, None, None, None, None)

In [19]:
d.backward(retain_graph=True) # fine
e.backward(retain_graph=True) # fine
d.backward() # also fine
e.backward() # error will occur!

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

# Train model

In [20]:
# Create random dataset. Every Dataset has to implement __len__ and __getitem__
class SyntheticDataset(Dataset):
    def __init__(self, num_samples=1000, input_dim=20):
        self.X = torch.rand(num_samples, input_dim)
        self.y = (torch.mean(self.X, dim=1) > 1/2).type(torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [21]:
# Create model
class MultiLayerNet(nn.Module):
    def __init__(self, input_dim=20, hidden_dims=[64, 32], output_dim=2):
        super(MultiLayerNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dims[0])
        self.bn1 = nn.BatchNorm1d(hidden_dims[0])
        self.activation1 = nn.ReLU()
            
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.bn2 = nn.BatchNorm1d(hidden_dims[1])
        self.activation2 = nn.ReLU()
            
        self.head = nn.Linear(hidden_dims[1], output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.activation1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.activation2(x)
        x = self.head(x)
        return x

In [22]:
# Training loop
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Train Loss: {avg_loss:.4f}")

In [23]:
# Validation loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100.0 * correct / total
    print(f"Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")
    return avg_loss, accuracy

In [28]:
# Hyperparameters and setup
input_dim = 20
batch_size = 32
epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset and split
dataset = SyntheticDataset(num_samples=1000, input_dim=input_dim)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [26]:
# Model, criterion, optimizer
model = MultiLayerNet(input_dim=input_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

In [29]:
class EarlyStopping:
    def __init__(self, patience=2, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.early_stop = False
    def __call__(self, train_loss, validation_loss):
        if (validation_loss - train_loss) > self.min_delta:
            self.counter +=1
            if self.counter >= self.patience:
                self.early_stop = True

In [35]:
# Full training + validation loop
prev_val_loss = float("inf")
early_stopping = EarlyStopping(patience=3, min_delta=0.1)
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    train(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    if prev_val_loss - val_loss > early_stopping.min_delta:
        early_stopping.counter = 0
    else:
        early_stopping.counter += 1
    if early_stopping.counter >= early_stopping.patience:
        early_stopping.early_stop = True
    if early_stopping.early_stop:
        print("Early stopping")
        break
    prev_val_loss = val_loss


Epoch 1/10
Train Loss: 0.0357
Validation Loss: 0.2303, Accuracy: 90.50%

Epoch 2/10
Train Loss: 0.0327
Validation Loss: 0.2387, Accuracy: 93.00%

Epoch 3/10
Train Loss: 0.0243
Validation Loss: 0.2486, Accuracy: 91.00%

Epoch 4/10
Train Loss: 0.0389
Validation Loss: 0.2616, Accuracy: 92.50%
Early stopping


# Tasks
- add early stopping
- play with number of parameters in each layer, activation function and regularization parameters and observe how the training changes